# Pump Failure Classifier Training

This notebook trains a 6-class XGBoost classifier to predict pump failure modes:
- NORMAL
- BEARING_WEAR
- CAVITATION
- VALVE_FAILURE
- OVERHEATING
- SEAL_LEAK

## Required Packages

Add these packages via the **Packages** dropdown in Snowsight:
- `snowflake-ml-python`
- `xgboost`
- `scikit-learn`
- `pandas`
- `numpy`

In [ ]:
# Import packages
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col
import pandas as pd
import numpy as np

session = get_active_session()
session.use_database("PDM_DEMO")
session.use_schema("ML")
print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

In [ ]:
# Define feature columns
FEATURE_COLS = [
    'SUCTION_PRESSURE', 'DISCHARGE_PRESSURE', 'FLOW_RATE', 'MOTOR_CURRENT',
    'PUMP_SPEED', 'BEARING_TEMP', 'CASING_TEMP', 'VIBRATION_RMS', 
    'VALVE_POSITION', 'LEAK_RATE', 'PRESSURE_DIFF', 'POWER_PROXY',
    'VIBRATION_1H_MEAN', 'VIBRATION_1H_STD', 'BEARING_TEMP_1H_MEAN', 
    'BEARING_TEMP_1H_STD', 'LEAK_RATE_1H_MEAN', 'LEAK_RATE_1H_STD',
    'SUCTION_1H_MEAN', 'SUCTION_1H_STD', 'FLOW_1H_MEAN', 'FLOW_1H_STD',
    'VIBRATION_24H_MEAN', 'VIBRATION_24H_STD', 'BEARING_TEMP_24H_MEAN',
    'BEARING_TEMP_24H_STD', 'LEAK_RATE_24H_MEAN', 'LEAK_RATE_24H_STD',
    'SUCTION_24H_MEAN', 'SUCTION_24H_STD', 'FLOW_24H_MEAN', 'FLOW_24H_STD',
    'VALVE_24H_MEAN', 'VALVE_24H_STD', 'CASING_TEMP_24H_MEAN',
    'MOTOR_CURRENT_24H_MEAN', 'VIBRATION_TREND_24H', 'TEMP_TREND_24H',
    'LEAK_TREND_24H', 'SUCTION_TREND_24H', 'FLOW_TREND_24H'
]

CLASS_LABELS = ['NORMAL', 'BEARING_WEAR', 'CAVITATION', 'OVERHEATING', 'SEAL_LEAK', 'VALVE_FAILURE']
print(f"Features: {len(FEATURE_COLS)}, Classes: {len(CLASS_LABELS)}")

In [ ]:
# Load training data
print("Loading training data...")
df = session.table("PUMP_TRAINING_DATA").filter(
    col("VIBRATION_TREND_24H").is_not_null()
).select(FEATURE_COLS + ['STATE', 'IS_TRAIN']).to_pandas()

print(f"Total rows: {len(df):,}")
print(f"\nClass distribution:")
print(df['STATE'].value_counts())

In [ ]:
# Split train/test and prepare features
from sklearn.preprocessing import LabelEncoder

train_df = df[df['IS_TRAIN'] == True].copy()
test_df = df[df['IS_TRAIN'] == False].copy()
print(f"Train: {len(train_df):,} rows, Test: {len(test_df):,} rows")

le = LabelEncoder()
le.fit(CLASS_LABELS)

X_train = train_df[FEATURE_COLS].fillna(0).values
y_train = le.transform(train_df['STATE'])
X_test = test_df[FEATURE_COLS].fillna(0).values
y_test = le.transform(test_df['STATE'])

In [ ]:
# Train XGBoost Classifier
import xgboost as xgb

print("Training XGBoost Classifier...")
clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    objective='multi:softprob',
    num_class=6,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print("Training complete!")

In [ ]:
# Evaluate model
from sklearn.metrics import classification_report, accuracy_score

y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_LABELS))

In [ ]:
# Feature importance
importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': clf.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Important Features:")
print(importance_df.head(10).to_string(index=False))

In [ ]:
# Register model in Snowflake Model Registry
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="PDM_DEMO", schema_name="ML")

sample_input = pd.DataFrame([X_train[0]], columns=FEATURE_COLS)

mv = reg.log_model(
    clf,
    model_name="PUMP_CLASSIFIER",
    version_name="V1",
    sample_input_data=sample_input,
    conda_dependencies=["xgboost", "scikit-learn"],
    target_platforms=["WAREHOUSE"],
    comment="6-class pump failure classifier"
)

print(f"Model registered: {mv.model_name} version {mv.version_name}")

In [ ]:
# Verify model functions
print("Model functions available:")
print(mv.show_functions())